# 🛡️ Complete NLP Master Notebook: SMS Spam Classifier & TF-IDF
### 100 Days of Machine Learning | Syed Muhammad Qamran
**Topic:** Natural Language Processing (NLP) from Scratch  
**Stack:** Python, Pandas, Scikit-Learn, Matplotlib, Seaborn

> **Self-Contained:** This notebook runs seamlessly in any Python environment (Local Jupyter, VS Code, Google Colab). If the dataset is not found locally, it automatically downloads it directly from GitHub!

---

### Notebook Sections:
1. **Environment Setup & Imports**
2. **Dataset Acquisition & Exploratory Data Analysis (EDA)**
3. **Text Preprocessing Pipeline from Scratch** (Entity Normalization, Stopwords, Stemming)
4. **Intuition: Bag of Words vs. TF-IDF vs. N-Grams** (Controlled Toy Example)
5. **Feature Extraction with Sublinear TF-IDF**
6. **Model Training** (Multinomial Naive Bayes, Regularized Logistic Regression, Random Forest)
7. **Performance Evaluation & Confusion Matrix Heatmaps** (The Precision vs. Recall Philosophy)
8. **Model Interpretability & Feature Importance** (Top Predictive Spam vs. Ham Terms)
9. **Live Interactive Inference & Token-Level Attribution**


## 1. Environment Setup & Imports


In [ ]:
# Core data manipulation and math
import os
import re
import string
import urllib.request
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn NLP and Machine Learning modules
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Visual styling
sns.set_theme(style="whitegrid")
%matplotlib inline

print("All libraries imported successfully!")


## 2. Dataset Acquisition & Exploratory Data Analysis (EDA)
We load the SMS Spam Collection dataset (5,574 messages).
If the local file is not found, the script automatically downloads it directly from GitHub.


In [ ]:
# Automatic dataset location / download
data_dir = "data"
data_file = os.path.join(data_dir, "sms_spam_collection.csv")

if not os.path.exists(data_file):
    os.makedirs(data_dir, exist_ok=True)
    url = "https://raw.githubusercontent.com/Syed-Muhammad-Qamran/sms-spam-classifier/main/data/sms_spam_collection.csv"
    print(f"Downloading dataset from {url}...")
    urllib.request.urlretrieve(url, data_file)
    print("Download complete!")

df = pd.read_csv(data_file)
print(f"Total SMS Messages Loaded: {len(df)}")
display(df.head(6))

# Class balance check
print("\nClass Distribution:")
counts = df['label'].value_counts()
pcts = df['label'].value_counts(normalize=True) * 100
summary_df = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts.round(2)})
display(summary_df)

# Visualizing class distribution
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.barplot(x=counts.index, y=counts.values, palette=['#10B981', '#EF4444'], ax=ax)
ax.set_title("Class Distribution: Ham vs. Spam", fontsize=13, fontweight='bold')
ax.set_ylabel("Number of Messages")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", 
                (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold', fontsize=12)
plt.show()

# Message length analysis (Character length & Word count)
df['char_length'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda msg: len(str(msg).split()))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
sns.histplot(data=df, x='char_length', hue='label', bins=50, kde=True, palette=['#10B981', '#EF4444'], ax=axes[0])
axes[0].set_title("Character Length Distribution (Spam is noticeably longer)")
axes[0].set_xlim(0, 300)

sns.histplot(data=df, x='word_count', hue='label', bins=30, kde=True, palette=['#10B981', '#EF4444'], ax=axes[1])
axes[1].set_title("Word Count Distribution")
axes[1].set_xlim(0, 60)

plt.tight_layout()
plt.show()


## 3. Text Preprocessing Pipeline from Scratch
Raw text messages contain noise: varied casing, punctuation, phone numbers, currencies, and URLs.
Our pipeline carries out:
1. **Lowercasing**
2. **Entity Normalization** (URLs $\rightarrow$ `urladdr`, Money $\rightarrow$ `moneynum`, Phone numbers $\rightarrow$ `phonenum`, Digits $\rightarrow$ `nummark`)
3. **Punctuation Removal**
4. **Stopword Filtering**
5. **Porter Suffix Stemming** (`running` $\rightarrow$ `run`, `congratulations` $\rightarrow$ `congratul`)


In [ ]:
# Self-contained stopwords set
STOPWORDS = {
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 'and', 'any', 'are', 'aren',
    "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but',
    'by', 'can', 'cannot', 'could', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does',
    'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further',
    'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', 'her',
    'here', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'i', 'if', 'in', 'into', 'is', 'isn', "isn't",
    'it', "it's", 'its', 'itself', 'just', 'll', 'm', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn',
    "mustn't", 'my', 'myself', 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or',
    'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she',
    "she's", 'should', 'shouldn', "shouldn't", 'so', 'some', 'such', 't', 'than', 'that', "that'll", 'the',
    'their', 'theirs', 'them', 'themselves', 'then', 'there', 'these', 'they', 'this', 'those', 'through',
    'to', 'too', 'under', 'until', 'up', 've', 'very', 'was', 'wasn', "wasn't", 'we', 'were', 'weren',
    "weren't", 'what', 'when', 'where', 'which', 'while', 'who', 'whom', 'why', 'will', 'with', 'won',
    "won't", 'wouldn', "wouldn't", 'y', 'you', "you'd", "you'll", "you're", "you've", 'your', 'yours',
    'yourself', 'yourselves'
}

class SimplePorterStemmer:
    """Suffix stripper inspired by Porter's stemmer algorithm."""
    @staticmethod
    def stem(word: str) -> str:
        word = word.lower().strip()
        if len(word) <= 3:
            return word

        if word.endswith('sses'):
            word = word[:-2]
        elif word.endswith('ies'):
            word = word[:-2]
        elif word.endswith('ss'):
            pass
        elif word.endswith('s') and not word.endswith('us') and not word.endswith('is'):
            word = word[:-1]

        if word.endswith('eed'):
            if len(word) > 4:
                word = word[:-1]
        elif word.endswith('ed') and len(word) > 4:
            word = word[:-2]
        elif word.endswith('ing') and len(word) > 5:
            word = word[:-3]

        if len(word) >= 3 and word[-1] == word[-2] and word[-1] in 'bdfgmnprt':
            word = word[:-1]

        suffixes = (
            ('ational', 'ate'), ('tional', 'tion'), ('ization', 'ize'),
            ('fulness', 'ful'), ('ousness', 'ous'), ('iveness', 'ive'),
            ('ment', ''), ('able', ''), ('ible', ''), ('ful', ''),
            ('ous', ''), ('ive', ''), ('ize', ''), ('ly', ''), ('e', '')
        )
        for old, new in suffixes:
            if word.endswith(old) and len(word) - len(old) >= 3:
                word = word[:-len(old)] + new
                break

        return word

def clean_text(text: str, stem: bool = True) -> str:
    """Full NLP text cleaning pipeline."""
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' urladdr ', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', ' emailaddr ', text)
    text = re.sub(r'[£$€]\s*\d+(?:\.\d+)?|\d+(?:\.\d+)?\s*[£$€]', ' moneynum ', text)
    text = re.sub(r'\b(?:\+?\d{1,3}[-.\s]?)?\(?\d{3,5}\)?[-.\s]?\d{3,5}[-.\s]?\d{3,5}\b', ' phonenum ', text)
    text = re.sub(r'\b\d+\b', ' nummark ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))

    tokens = text.split()
    stemmer = SimplePorterStemmer()

    cleaned = []
    for t in tokens:
        if t not in STOPWORDS and len(t) > 1:
            cleaned.append(stemmer.stem(t) if stem else t)

    return " ".join(cleaned)

# Demonstration before and after
demo_msg = "URGENT! You have won £2,000 cash! Call 0800123456 or click http://win.com now!"
print(f"Raw:     {demo_msg}")
print(f"Cleaned: {clean_text(demo_msg)}")

# Apply cleaning across entire dataset
print("\nCleaning all 5,574 dataset messages...")
df['target'] = df['label'].str.lower().map({'ham': 0, 'spam': 1})
df['cleaned'] = df['message'].apply(clean_text)
df_clean = df[df['cleaned'].str.len() > 0].copy().reset_index(drop=True)
print(f"Ready! Total clean messages: {len(df_clean)}")
display(df_clean[['label', 'message', 'cleaned']].head(4))


## 4. Intuition: Bag of Words vs. TF-IDF vs. N-Grams
Before vectorizing the full dataset, let's understand the mathematics on a controlled 4-sentence corpus:
- **CountVectorizer (Bag of Words):** Simply counts how many times word $t$ appears.
- **TF-IDF:** Multiplies term frequency by inverse document frequency to discount common conversational words and amplify distinctive keywords.
- **N-Grams:** Extracts word pairs (`cash prize`, `secret prize`) to preserve multi-word context.


In [ ]:
toy_corpus = [
    "the secret prize is waiting for you",
    "you have won a cash prize and cash bonus",
    "call me tomorrow to discuss the project",
    "are you free for project lunch tomorrow"
]
toy_labels = ["Doc 0 (Spam)", "Doc 1 (Spam)", "Doc 2 (Ham)", "Doc 3 (Ham)"]

# 1. Bag of Words
bow = CountVectorizer(stop_words='english')
bow_mat = bow.fit_transform(toy_corpus)
df_toy_bow = pd.DataFrame(bow_mat.toarray(), index=toy_labels, columns=bow.get_feature_names_out())
print("=== BAG OF WORDS (Counts) ===")
display(df_toy_bow)

# 2. TF-IDF with Unigrams and Bigrams
tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
tfidf_mat = tfidf.fit_transform(toy_corpus)
df_toy_tfidf = pd.DataFrame(np.round(tfidf_mat.toarray(), 2), index=toy_labels, columns=tfidf.get_feature_names_out())
print("\n=== TF-IDF WITH N-GRAMS (Weights) ===")
display(df_toy_tfidf)


## 5. Feature Extraction on the Full Dataset
We use stratified sampling (80% train / 20% test) to preserve the 86.6% Ham / 13.4% Spam proportion, then fit the `TfidfVectorizer` with:
- `max_features=3000`
- `ngram_range=(1, 2)` (capturing both single words and two-word phrases)
- `sublinear_tf=True` (replaces $\text{TF}$ with $1 + \log(\text{TF})$)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_clean['cleaned'],
    df_clean['target'],
    test_size=0.20,
    random_state=42,
    stratify=df_clean['target']
)

print(f"Train samples: {len(X_train)} ({sum(y_train==0)} Ham, {sum(y_train==1)} Spam)")
print(f"Test samples:  {len(X_test)} ({sum(y_test==0)} Ham, {sum(y_test==1)} Spam)")

vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
feature_names = np.array(vectorizer.get_feature_names_out())

print(f"\nTF-IDF Matrix Shape: {X_train_vec.shape}")
print(f"Vocabulary Dimension: {len(feature_names)} features")
print(f"Sparsity: {(1.0 - (X_train_vec.nnz / (X_train_vec.shape[0] * X_train_vec.shape[1]))) * 100:.2f}%")


## 6. Model Training & Comparison
We train three complementary machine learning classifiers:
1. **Multinomial Naive Bayes (`MultinomialNB`)**: Probabilistic text baseline with Laplace smoothing ($\alpha=1.0$).
2. **Regularized Logistic Regression (`LogisticRegression`)**: Linear classifier with $L_2$ penalty ($C=2.0$).
3. **Random Forest (`RandomForestClassifier`)**: Ensemble of 100 decision trees.


In [ ]:
models = {
    "Multinomial Naive Bayes": MultinomialNB(alpha=1.0),
    "Logistic Regression": LogisticRegression(C=2.0, max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=25, random_state=42)
}

trained_models = {}
metrics_dict = {}

for name, clf in models.items():
    print(f"Training {name}...")
    clf.fit(X_train_vec, y_train)
    y_pred = clf.predict(X_test_vec)
    
    cm = confusion_matrix(y_test, y_pred)
    metrics_dict[name] = {
        "Accuracy (%)": round(accuracy_score(y_test, y_pred) * 100, 2),
        "Precision (%)": round(precision_score(y_test, y_pred, zero_division=0) * 100, 2),
        "Recall (%)": round(recall_score(y_test, y_pred, zero_division=0) * 100, 2),
        "F1-Score": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "False Positives (FP)": int(cm[0][1]),
        "False Negatives (FN)": int(cm[1][0]),
        "Confusion Matrix": cm
    }
    trained_models[name] = clf

# Metrics comparison table
results_summary = pd.DataFrame([
    {
        "Model": k,
        "Accuracy": f"{v['Accuracy (%)']} %",
        "Precision (Spam)": f"{v['Precision (%)']} %",
        "Recall (Spam)": f"{v['Recall (%)']} %",
        "F1-Score": v['F1-Score'],
        "False Positives": v['False Positives (FP)'],
        "False Negatives": v['False Negatives (FN)']
    }
    for k, v in metrics_dict.items()
])
display(results_summary)


## 7. Performance Evaluation & Confusion Matrix Heatmaps

### The Core Spam Metric: Precision vs. Recall
In spam filtering:
- **False Positive (FP):** Legitimate email sent to Spam folder $\rightarrow$ **Catastrophic!** You miss a bank fraud alert or job offer.
- **False Negative (FN):** Spam email sent to Inbox $\rightarrow$ **Minor annoyance.** You simply swipe delete.

**Multinomial Naive Bayes and Random Forest achieved 99-100% Precision (0 to 1 False Positives out of 965 legitimate messages)!**


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

for idx, (m_name, res) in enumerate(metrics_dict.items()):
    cm = res["Confusion Matrix"]
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False,
        xticklabels=["Predicted Ham", "Predicted Spam"],
        yticklabels=["Actual Ham", "Actual Spam"],
        ax=axes[idx], annot_kws={"size": 13, "weight": "bold"}
    )
    axes[idx].set_title(m_name, fontsize=12, fontweight="bold", pad=10)

plt.tight_layout()
plt.show()


## 8. Model Interpretability & Feature Importance
Which words actually signal spam versus legitimate conversation?
In Naive Bayes, each feature $w$ has conditional log-probabilities $\log P(w | \text{spam})$ and $\log P(w | \text{ham})$.
The difference:
$$\Delta(w) = \log P(w \mid \text{spam}) - \log P(w \mid \text{ham})$$
quantifies the log-odds advantage of word $w$.


In [ ]:
nb = trained_models["Multinomial Naive Bayes"]
nb_diff = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]

# Top 20 spam features
top_spam_idx = np.argsort(nb_diff)[-20:][::-1]
df_top_spam = pd.DataFrame({
    'Keyword': feature_names[top_spam_idx],
    'Log-Odds Advantage': nb_diff[top_spam_idx]
})

# Top 20 ham features
top_ham_idx = np.argsort(nb_diff)[:20]
df_top_ham = pd.DataFrame({
    'Keyword': feature_names[top_ham_idx],
    'Log-Odds Advantage': np.abs(nb_diff[top_ham_idx])
})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.barplot(data=df_top_spam, x='Log-Odds Advantage', y='Keyword', color='#EF4444', ax=axes[0])
axes[0].set_title("Top 20 Spam-Predictive Keywords", fontsize=13, fontweight='bold', color='#EF4444')

sns.barplot(data=df_top_ham, x='Log-Odds Advantage', y='Keyword', color='#10B981', ax=axes[1])
axes[1].set_title("Top 20 Ham-Predictive Keywords", fontsize=13, fontweight='bold', color='#10B981')

plt.tight_layout()
plt.show()


## 9. Live Interactive Inference & Word-Level Attribution


In [ ]:
vocab_map = {name: idx for idx, name in enumerate(feature_names)}

def predict_message(message: str, model_name: str = "Multinomial Naive Bayes"):
    """Predicts Ham or Spam and prints token-level attribution breakdown."""
    model = trained_models[model_name]
    
    # Preprocess & Vectorize
    cleaned = clean_text(message)
    if not cleaned.strip():
        print("Empty message!")
        return
        
    vec = vectorizer.transform([cleaned])
    probs = model.predict_proba(vec)[0]
    p_ham, p_spam = probs[0], probs[1]
    verdict = "🚨 SPAM" if p_spam >= 0.50 else "✅ HAM"
    
    print("=" * 65)
    print(f"INPUT:   {message}")
    print(f"CLEANED: {cleaned}")
    print(f"VERDICT: {verdict} (Spam: {p_spam*100:.2f}%, Ham: {p_ham*100:.2f}%)")
    print("\nTOKEN ATTRIBUTION BREAKDOWN:")
    for token in cleaned.split():
        if token in vocab_map:
            score = nb_diff[vocab_map[token]]
            tag = "[SPAM SIGNAL]" if score > 0.5 else ("[HAM SIGNAL]" if score < -0.5 else "[NEUTRAL]")
            print(f"  • {token:<15} score: {score:+.3f}  {tag}")
        else:
            print(f"  • {token:<15} [OUT OF VOCABULARY]")
    print("=" * 65 + "\n")

# Test Cases:
# Case 1: Urgent lottery scam
predict_message("URGENT! You have won £2,000 cash prize. Call 0800123456 now or text CLAIM to 88888!")

# Case 2: Normal conversation
predict_message("Hey mom, are we having dinner at home tonight? Let me know.")

# Case 3: Ambiguous context testing the word 'free'
predict_message("Are you free tonight to grab some pizza and coffee after work?")
